# Enrichissement Automatique de l'Ontologie CVDO
## Avec alignement sémantique multi-niveaux (approche Wu et al. 2025)

Ce notebook intègre les triplets validés dans l'ontologie CVDO.
L'alignement des entités extraites avec les classes existantes est réalisé
selon l'approche de Wu et al. (2025) en 3 niveaux :
1. Matching exact (insensible à la casse)
2. Matching avec nettoyage des suffixes ontologiques (disease), (process)
3. Similarité cosinus BioBERT >= 0.98 (embeddings normalisés)

In [1]:
!pip install -q rdflib pandas sentence-transformers

import pandas as pd
import numpy as np
import os
from datetime import datetime
from rdflib import Graph, Namespace, URIRef, Literal, BNode
from rdflib.namespace import RDF, RDFS, OWL, XSD
from sentence_transformers import SentenceTransformer

print('Chargement de BioBERT pour l\'alignement sémantique...')
os.environ["TOKENIZERS_PARALLELISM"] = "false"
embedder = SentenceTransformer("dmis-lab/biobert-base-cased-v1.2")
print('tout est ok')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.4/615.4 kB 9.4 MB/s eta 0:00:00
Chargement de BioBERT pour l'alignement sémantique...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

BertModel LOAD REPORT from: dmis-lab/biobert-base-cased-v1.2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


vocab.txt: 0.00B [00:00, ?B/s]

tout est ok


In [2]:
# ============================================================================
# NAMESPACES + MAPPING RELATIONS
# ============================================================================

CVDO = Namespace("http://cvdo.org/")
SNOMED = Namespace("http://snomed.info/id/")
BFO = Namespace("http://purl.obolibrary.org/obo/")
DC = Namespace("http://purl.org/dc/elements/1.1/")
SKOS = Namespace("http://www.w3.org/2004/02/skos/core#")

RELATION_TO_PROPERTY = {
    'est_facteur_de_risque_de': {
        'uri': CVDO.hasRiskFactor,
        'label': 'has risk factor',
        'snomed': '405506007',
        'domain_comment': 'Relates a cardiovascular condition to its risk factor',
    },
    'est_traite_par': {
        'uri': CVDO.hasTreatment,
        'label': 'has treatment',
        'snomed': '116681000007109',
        'domain_comment': 'Relates a condition to its therapeutic intervention',
    },
    'est_symptome_de': {
        'uri': CVDO.hasSymptom,
        'label': 'has symptom',
        'snomed': '363270007',
        'domain_comment': 'Relates a condition to its clinical manifestation',
    },
    'est_complication_de': {
        'uri': CVDO.hasComplication,
        'label': 'has complication',
        'snomed': '408729004',
        'domain_comment': 'Relates a condition to its complication',
    },
    'est_surveille_par': {
        'uri': CVDO.isMonitoredBy,
        'label': 'is monitored by',
        'snomed': '363713009',
        'domain_comment': 'Relates a condition to its monitoring method',
    },
}

print('nickel')

nickel


In [ ]:
# ============================================================================
# ALIGNEMENT SÉMANTIQUE MULTI-NIVEAUX (approche Wu et al. 2025)
# ============================================================================
#
# Wu et al. (2025) résolvent l'alignement en combinant :
# 1. Matching exact (synonymes, normalisation)
# 2. Similarité sémantique (embeddings biomédicaux)
# 3. Règles de priorité (arbitrage des conflits)
#
# Notre implémentation adapte cette approche en 3 niveaux :
# Niveau 1 : Matching exact insensible à la casse
# Niveau 2 : Matching avec nettoyage des suffixes ontologiques (disease), (process)
# Niveau 3 : Similarité cosinus BioBERT avec seuil strict (>= 0.98)
# ============================================================================

SEUIL_SIMILARITE = 0.98

# Variables globales pour l'index sémantique
cvdo_label_texts = []
cvdo_label_uris = []
cvdo_embeddings = None
cvdo_labels_exact = {}      # label.lower() -> URI
cvdo_labels_nettoye = {}    # label nettoyé sans suffixes -> URI


def construire_index_semantique(g):
    """
    Construit un index d'alignement multi-niveaux :
    - Index exact (matching par string)
    - Index nettoyé (sans suffixes ontologiques)
    - Index sémantique BioBERT (similarité cosinus normalisée)
    """
    global cvdo_label_texts, cvdo_label_uris, cvdo_embeddings
    global cvdo_labels_exact, cvdo_labels_nettoye

    for s in g.subjects(RDF.type, OWL.Class):
        for l in g.objects(s, RDFS.label):
            label = str(l).strip()
            if len(label) > 2:
                uri = str(s)

                # Index exact
                cvdo_labels_exact[label.lower()] = uri

                # Index nettoyé : retirer les suffixes (disease), (process), etc.
                label_clean = label.lower().strip()
                for suffixe in [' (disease)', ' (process)', ' (disorder)']:
                    if label_clean not in cvdo_labels_nettoye or '(disease)' in label.lower():
                          cvdo_labels_nettoye[label_clean] = uri
                cvdo_labels_nettoye[label_clean] = uri

                # Pour l'index sémantique
                cvdo_label_texts.append(label)
                cvdo_label_uris.append(uri)

    # Encoder avec BioBERT + normalisation L2
    print(f'  Encodage de {len(cvdo_label_texts)} labels avec BioBERT...')
    cvdo_embeddings = embedder.encode(cvdo_label_texts, show_progress_bar=False)
    cvdo_embeddings = cvdo_embeddings / np.linalg.norm(cvdo_embeddings, axis=1, keepdims=True)
    print(f'  ✓ Index sémantique construit ({cvdo_embeddings.shape})')


def trouver_classe_existante(g, label_cherche):
    """
    Alignement multi-niveaux inspiré de Wu et al. (2025) :

    Niveau 1 : Matching exact (insensible à la casse)
    Niveau 2 : Matching avec nettoyage des suffixes (disease)/(process)
    Niveau 3 : Similarité cosinus BioBERT >= 0.98

    Retourne (URI, label_matchée, niveau_match) ou (None, None, None)
    """
    global cvdo_embeddings, cvdo_label_texts, cvdo_label_uris
    global cvdo_labels_exact, cvdo_labels_nettoye

    label_lower = label_cherche.lower().strip()

    # Niveau 1 : Matching exact
    if label_lower in cvdo_labels_exact:
        uri = cvdo_labels_exact[label_lower]
        return URIRef(uri), label_lower, "exact"

    # Niveau 2 : Matching nettoyé
    if label_lower in cvdo_labels_nettoye:
        uri = cvdo_labels_nettoye[label_lower]
        label_orig = next((t for t in cvdo_label_texts if cvdo_labels_exact.get(t.lower()) == uri), label_lower)
        return URIRef(uri), label_orig, "nettoyé"

    # Niveau 3 : Similarité cosinus BioBERT (normalisé)
    if cvdo_embeddings is not None and len(cvdo_label_texts) > 0:
        entite_emb = embedder.encode([label_cherche])
        entite_emb = entite_emb / np.linalg.norm(entite_emb, axis=1, keepdims=True)
        similarities = np.dot(cvdo_embeddings, entite_emb.T).flatten()
        best_idx = int(np.argmax(similarities))
        best_score = float(similarities[best_idx])

        if best_score >= SEUIL_SIMILARITE:
            return URIRef(cvdo_label_uris[best_idx]), cvdo_label_texts[best_idx], f"BioBERT ({best_score:.3f})"

    return None, None, None


def nettoyer_nom_uri(nom):
    """Transforme un nom d'entité en fragment URI valide."""
    return (nom.strip()
            .replace(' ', '_')
            .replace('-', '_')
            .replace('(', '')
            .replace(')', '')
            .replace('/', '_'))


def creer_classe_owl(g, nom_entite, parent=None):
    """
    Crée une nouvelle classe OWL ou réutilise une existante.
    Utilise l'alignement multi-niveaux (Wu et al. 2025).
    Retourne (uri, est_nouvelle, label_matchée, niveau_match).
    """
    classe_existante, label_match, niveau = trouver_classe_existante(g, nom_entite)
    if classe_existante:
        return classe_existante, False, label_match, niveau

    # Pas de match : créer une nouvelle classe
    uri_fragment = nettoyer_nom_uri(nom_entite)
    classe_uri = CVDO[uri_fragment]

    g.add((classe_uri, RDF.type, OWL.Class))
    g.add((classe_uri, RDFS.label, Literal(nom_entite, lang="en")))

    if parent:
        g.add((classe_uri, RDFS.subClassOf, parent))

    g.add((classe_uri, DC.source, Literal("Auto-generated by RAG+LLM pipeline (XP2)")))
    g.add((classe_uri, DC.date, Literal(datetime.now().strftime("%Y-%m-%d"))))

    # Ajouter au cache pour les entités suivantes
    global cvdo_label_texts, cvdo_label_uris, cvdo_embeddings
    global cvdo_labels_exact, cvdo_labels_nettoye
    new_emb = embedder.encode([nom_entite])
    new_emb = new_emb / np.linalg.norm(new_emb, axis=1, keepdims=True)
    cvdo_label_texts.append(nom_entite)
    cvdo_label_uris.append(str(classe_uri))
    cvdo_embeddings = np.vstack([cvdo_embeddings, new_emb])
    cvdo_labels_exact[nom_entite.lower()] = str(classe_uri)
    cvdo_labels_nettoye[nom_entite.lower()] = str(classe_uri)

    return classe_uri, True, None, None


def creer_object_properties(g):
    """
    Crée les 5 object properties cardiovasculaires
    si elles n'existent pas déjà dans CVDO.
    Chaque propriété est alignée sur SNOMED-CT via skos:exactMatch.
    """
    properties_creees = []

    for relation, info in RELATION_TO_PROPERTY.items():
        prop_uri = info['uri']

        if (prop_uri, RDF.type, OWL.ObjectProperty) in g:
            print(f"  ✓ Propriété existante : {info['label']}")
            continue

        g.add((prop_uri, RDF.type, OWL.ObjectProperty))
        g.add((prop_uri, RDFS.label, Literal(info['label'], lang="en")))
        g.add((prop_uri, RDFS.comment, Literal(info['domain_comment'], lang="en")))
        g.add((prop_uri, SKOS.exactMatch, SNOMED[info['snomed']]))
        g.add((prop_uri, DC.source, Literal(f"SNOMED-CT: {info['snomed']}")))

        properties_creees.append(info['label'])
        print(f"  + Propriété créée : {info['label']} (SNOMED: {info['snomed']})")

    return properties_creees


def ajouter_axiome_restriction(g, classe_sujet, prop_uri, classe_objet):
    """
    Ajoute un axiome OWL :
    classe_sujet SubClassOf (prop_uri some classe_objet)

    Pattern owl:Restriction + owl:someValuesFrom
    compatible avec le raisonneur HermiT de Protégé.
    """
    restriction = BNode()
    g.add((restriction, RDF.type, OWL.Restriction))
    g.add((restriction, OWL.onProperty, prop_uri))
    g.add((restriction, OWL.someValuesFrom, classe_objet))
    g.add((classe_sujet, RDFS.subClassOf, restriction))


print('nickel')

nickel


In [4]:
from google.colab import files

print('Uploadez cvdo.owl (ontologie originale) et le CSV des triplets...')
uploaded = files.upload()

for f in uploaded.keys():
    print(f'  ✓ {f}')

CHEMIN_ONTOLOGIE = [f for f in uploaded.keys() if f.endswith('.owl')][0]
CHEMIN_CSV = [f for f in uploaded.keys() if f.endswith('.csv')][0]

print(f'\nOntologie : {CHEMIN_ONTOLOGIE}')
print(f'Triplets  : {CHEMIN_CSV}')

Uploadez cvdo.owl (ontologie originale) et le CSV des triplets...


Saving cvdoc.owl to cvdoc.owl
Saving XP2_triplets_c.csv to XP2_triplets_c.csv
  ✓ cvdoc.owl
  ✓ XP2_triplets_c.csv

Ontologie : cvdoc.owl
Triplets  : XP2_triplets_c.csv


In [5]:
print('[1/5] Chargement de l\'ontologie CVDO...')

g = Graph()
g.parse(CHEMIN_ONTOLOGIE, format="xml")

g.bind("cvdo", CVDO)
g.bind("snomed", SNOMED)
g.bind("bfo", BFO)
g.bind("dc", DC)
g.bind("skos", SKOS)
g.bind("owl", OWL)

nb_triples_avant = len(g)
nb_classes_avant = len(list(g.triples((None, RDF.type, OWL.Class))))
nb_props_avant = len(list(g.triples((None, RDF.type, OWL.ObjectProperty))))

print(f'  ✓ {nb_triples_avant} triples RDF')
print(f'  ✓ {nb_classes_avant} classes OWL')
print(f'  ✓ {nb_props_avant} Object Properties')

print('\n  Construction de l\'index d\'alignement multi-niveaux...')
construire_index_semantique(g)
print(f'  ✓ Index prêt : {len(cvdo_label_texts)} labels encodés')

[1/5] Chargement de l'ontologie CVDO...
  ✓ 5692 triples RDF
  ✓ 853 classes OWL
  ✓ 9 Object Properties

  Construction de l'index d'alignement multi-niveaux...
  Encodage de 295 labels avec BioBERT...
  ✓ Index sémantique construit ((295, 768))
  ✓ Index prêt : 295 labels encodés


In [6]:
print('[2/5] Création des Object Properties...')
props_creees = creer_object_properties(g)
print(f'\n  → {len(props_creees)} nouvelles propriétés ajoutées')

[2/5] Création des Object Properties...
  + Propriété créée : has risk factor (SNOMED: 405506007)
  + Propriété créée : has treatment (SNOMED: 116681000007109)
  + Propriété créée : has symptom (SNOMED: 363270007)
  + Propriété créée : has complication (SNOMED: 408729004)
  + Propriété créée : is monitored by (SNOMED: 363713009)

  → 5 nouvelles propriétés ajoutées


In [7]:
print('[3/5] Chargement des triplets validés...')

df = pd.read_csv(CHEMIN_CSV)
print(f'  ✓ {len(df)} triplets chargés')

# Retirer les triplets auto-référentiels
before = len(df)
df = df[df['sujet_normalise'].str.lower().str.strip() != df['objet_normalise'].str.lower().str.strip()]
if len(df) < before:
    print(f'  ⚠ {before - len(df)} triplets auto-référentiels retirés')

# Dédupliquer
before = len(df)
df = df.drop_duplicates(subset=['sujet_normalise', 'relation', 'objet_normalise'])
if len(df) < before:
    print(f'  ⚠ {before - len(df)} doublons retirés')

print(f'  → {len(df)} triplets uniques à intégrer')

print('\nAperçu des 10 premiers triplets :')
for _, row in df.head(10).iterrows():
    print(f"  {row['sujet_normalise']}  —[{row['relation']}]→  {row['objet_normalise']}  ({row['consensus']})")

[3/5] Chargement des triplets validés...
  ✓ 126 triplets chargés
  → 126 triplets uniques à intégrer

Aperçu des 10 premiers triplets :
  hypertension  —[est_complication_de]→  hypertensive urgencies  (2/3)
  hypertension  —[est_complication_de]→  chronic kidney disease  (2/3)
  hypertension  —[est_traite_par]→  renal denervation  (2/3)
  hypertension  —[est_traite_par]→  triple combination antihypertensive medication  (2/3)
  hypertension  —[est_traite_par]→  polypills  (2/3)
  hypertension  —[est_surveille_par]→  ambulatory blood pressure monitoring  (3/3)
  hypertension  —[est_surveille_par]→  blood pressure monitoring  (2/3)
  resistant hypertension  —[est_traite_par]→  renal denervation  (2/3)
  hypertension  —[est_facteur_de_risque_de]→  cardiovascular disease  (2/3)
  hypertension  —[est_complication_de]→  hypertension-mediated organ damage  (2/3)


In [8]:
print('[4/5] Intégration des triplets dans CVDO...')
print(f'      (alignement multi-niveaux Wu et al. 2025, seuil BioBERT = {SEUIL_SIMILARITE})')

nouvelles_classes = set()
classes_alignees_cvdo = {}  # label_extrait -> (label_cvdo, niveau)
classes_deja_vues = set()   # entités déjà traitées
axiomes_ajoutes = 0
erreurs = []

for idx, row in df.iterrows():
    sujet = str(row['sujet_normalise']).strip()
    relation = str(row['relation']).strip()
    objet = str(row['objet_normalise']).strip()
    consensus = str(row.get('consensus', '?'))

    if relation not in RELATION_TO_PROPERTY:
        erreurs.append(f"Relation inconnue: {relation} ({sujet} → {objet})")
        continue

    prop_info = RELATION_TO_PROPERTY[relation]

    # Créer ou trouver la classe sujet
    classe_sujet, est_nouvelle_s, match_label_s, niveau_s = creer_classe_owl(g, sujet)
    if sujet.lower() not in classes_deja_vues:
        classes_deja_vues.add(sujet.lower())
        if est_nouvelle_s:
            nouvelles_classes.add(sujet)
        else:
            classes_alignees_cvdo[sujet] = (match_label_s, niveau_s)

    # Créer ou trouver la classe objet
    classe_objet, est_nouvelle_o, match_label_o, niveau_o = creer_classe_owl(g, objet)
    if objet.lower() not in classes_deja_vues:
        classes_deja_vues.add(objet.lower())
        if est_nouvelle_o:
            nouvelles_classes.add(objet)
        else:
            classes_alignees_cvdo[objet] = (match_label_o, niveau_o)

    # Ajouter l'axiome SubClassOf avec restriction OWL
    ajouter_axiome_restriction(g, classe_sujet, prop_info['uri'], classe_objet)

    # Annoter avec la provenance
    g.add((classe_sujet, CVDO.enrichmentSource,
           Literal(f"ESC Guidelines - RAG+LLM consensus {consensus}")))
    g.add((classe_sujet, CVDO.enrichmentDate,
           Literal(datetime.now().strftime("%Y-%m-%d"), datatype=XSD.date)))

    axiomes_ajoutes += 1

print(f'  ✓ {axiomes_ajoutes} axiomes SubClassOf ajoutés')
print(f'  ✓ {len(nouvelles_classes)} nouvelles classes créées')
print(f'  ✓ {len(classes_alignees_cvdo)} classes existantes réutilisées (alignement multi-niveaux)')
print(f'  ✓ {len(classes_deja_vues)} entités uniques au total')

if erreurs:
    print(f'  ⚠ {len(erreurs)} erreurs :')
    for e in erreurs:
        print(f'    - {e}')

[4/5] Intégration des triplets dans CVDO...
      (alignement multi-niveaux Wu et al. 2025, seuil BioBERT = 0.98)
  ✓ 126 axiomes SubClassOf ajoutés
  ✓ 93 nouvelles classes créées
  ✓ 6 classes existantes réutilisées (alignement multi-niveaux)
  ✓ 99 entités uniques au total


In [9]:
OUTPUT = "cvdo_enrichie.owl"

print('[5/5] Export de l\'ontologie enrichie...')

g.serialize(destination=OUTPUT, format="xml")
print(f'  ✓ Fichier sauvegardé : {OUTPUT}')

nb_triples_apres = len(g)
nb_classes_apres = len(list(g.triples((None, RDF.type, OWL.Class))))
nb_props_apres = len(list(g.triples((None, RDF.type, OWL.ObjectProperty))))

print('\n' + '=' * 70)
print('RAPPORT D\'ENRICHISSEMENT')
print('=' * 70)
print(f'  Triples RDF      : {nb_triples_avant} → {nb_triples_apres} (+{nb_triples_apres - nb_triples_avant})')
print(f'  Classes OWL      : {nb_classes_avant} → {nb_classes_apres} (+{nb_classes_apres - nb_classes_avant})')
print(f'  Object Properties: {nb_props_avant} → {nb_props_apres} (+{nb_props_apres - nb_props_avant})')
print(f'  Axiomes ajoutés  : {axiomes_ajoutes}')
print(f'  Nouvelles classes : {len(nouvelles_classes)}')
print(f'  Classes alignées  : {len(classes_alignees_cvdo)}')
print(f'  Entités uniques   : {len(classes_deja_vues)}')
print(f'  Seuil BioBERT     : {SEUIL_SIMILARITE}')
print('=' * 70)
print(f'\n→ Ouvrir \'{OUTPUT}\' dans Protégé')
print('→ Lancer HermiT (Reasoner → HermiT → Start Reasoner)')
print('→ Vérifier qu\'aucune inconsistance n\'est détectée')

[5/5] Export de l'ontologie enrichie...
  ✓ Fichier sauvegardé : cvdo_enrichie.owl

RAPPORT D'ENRICHISSEMENT
  Triples RDF      : 5692 → 6668 (+976)
  Classes OWL      : 853 → 946 (+93)
  Object Properties: 9 → 14 (+5)
  Axiomes ajoutés  : 126
  Nouvelles classes : 93
  Classes alignées  : 6
  Entités uniques   : 99
  Seuil BioBERT     : 0.98

→ Ouvrir 'cvdo_enrichie.owl' dans Protégé
→ Lancer HermiT (Reasoner → HermiT → Start Reasoner)
→ Vérifier qu'aucune inconsistance n'est détectée


In [10]:
print('CLASSES ALIGNÉES (existantes réutilisées) :')
print('-' * 60)
for extrait, (cvdo_label, niveau) in sorted(classes_alignees_cvdo.items()):
    print(f'  ✓ \'{extrait}\' → \'{cvdo_label}\' [{niveau}]')

print(f'\nNOUVELLES CLASSES CRÉÉES :')
print('-' * 60)
for c in sorted(nouvelles_classes):
    print(f'  + {c}')

CLASSES ALIGNÉES (existantes réutilisées) :
------------------------------------------------------------
  ✓ 'acute ischaemic stroke' → 'ischaemic stroke' [BioBERT (0.981)]
  ✓ 'antiplatelet treatment' → 'antiplatelet therapy' [BioBERT (0.988)]
  ✓ 'coronary artery disease' → 'coronary heart disease' [BioBERT (0.984)]
  ✓ 'hypertrophic cardiomyopathy' → 'genetic hypertrophic cardiomyopathy' [BioBERT (0.988)]
  ✓ 'persistent atrial fibrillation' → 'persistent atrial fibrillation' [exact]
  ✓ 'thromboembolism' → 'thromboembolism' [exact]

NOUVELLES CLASSES CRÉÉES :
------------------------------------------------------------
  + BP-lowering interventions
  + DC cardioversion
  + Holter ECG
  + abdominal fatness
  + ablation
  + ablation-based rhythm-control
  + abnormal glucose regulation
  + acute coronary syndrome
  + aerobic exercise
  + ambulatory blood pressure monitoring
  + amiodarone
  + antiplatelet therapy
  + antithrombotic therapy
  + aortic ectasia
  + atrial fibrillation
  

In [11]:
print('DISTRIBUTION DES RELATIONS :')
print('-' * 40)
for rel, count in df['relation'].value_counts().items():
    prop_label = RELATION_TO_PROPERTY.get(rel, {}).get('label', rel)
    print(f'  {prop_label:30s} : {count} axiomes')

DISTRIBUTION DES RELATIONS :
----------------------------------------
  has treatment                  : 45 axiomes
  has risk factor                : 45 axiomes
  has complication               : 20 axiomes
  is monitored by                : 15 axiomes
  has symptom                    : 1 axiomes


In [12]:
print('EXEMPLES D\'AXIOMES AJOUTÉS (5 premiers) :')
print('-' * 60)

count = 0
for s, p, o in g.triples((None, OWL.someValuesFrom, None)):
    for prop in g.objects(s, OWL.onProperty):
        for classe in g.subjects(RDFS.subClassOf, s):
            label_s = g.value(classe, RDFS.label) or classe
            label_o = g.value(o, RDFS.label) or o
            label_p = g.value(prop, RDFS.label) or prop
            print(f'  {label_s}  —[{label_p}]→  {label_o}')
            count += 1
            if count >= 5:
                break
        if count >= 5:
            break
    if count >= 5:
        break

print(f'\n  ... et {axiomes_ajoutes - 5} autres axiomes')

EXEMPLES D'AXIOMES AJOUTÉS (5 premiers) :
------------------------------------------------------------
  heart valve improper close  —[http://purl.obolibrary.org/obo/BFO_0000066]→  http://purl.obolibrary.org/obo/FMA_7088
  cardiac blood circulation pathological process  —[http://purl.obolibrary.org/obo/BFO_0000066]→  http://purl.obolibrary.org/obo/FMA_7088
  heart valve improper opening  —[http://purl.obolibrary.org/obo/BFO_0000066]→  http://purl.obolibrary.org/obo/FMA_7088
  heart valve leaflet displacement into a cardiac chamber  —[http://purl.obolibrary.org/obo/BFO_0000066]→  http://purl.obolibrary.org/obo/FMA_7088
  http://purl.obolibrary.org/obo/DOID_9768  —[http://purl.obolibrary.org/obo/BFO_0000082]→  http://purl.obolibrary.org/obo/FMA_7088

  ... et 121 autres axiomes


In [13]:
from google.colab import files
files.download(OUTPUT)
print(f'✓ {OUTPUT} téléchargé')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✓ cvdo_enrichie.owl téléchargé
